In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import time

In [5]:
df = pd.read_parquet("hf://datasets/badrex/LLM-generated-emoji-descriptions/data/train-00000-of-00001.parquet")
df.head()

,character,unicode,short description,tags,LLM description
0,🥇,U+1F947,1ST PLACE MEDAL,"[first place, victory, achievement, success, c...","This emoji represents a first place medal, oft..."
1,🥈,U+1F948,2ND PLACE MEDAL,"[medal, silver, second place, achievement, suc...","This emoji represents a silver medal, often us..."
2,🥉,U+1F949,3RD PLACE MEDAL,"[medal, bronze, third place, achievement, spor...","This emoji represents a bronze medal, symboliz..."
3,🆎,U+1F18E,AB BUTTON (BLOOD TYPE),"[blood type, AB, medical, compatibility, trans...",This emoji represents the AB blood type symbol...
4,🏧,U+1F3E7,ATM SIGN,"[ATM, banking, finance, money, transaction, lo...","This emoji represents an ATM sign, often used ..."


In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [9]:
def form_prompts():
    result = []
    for index, row in df.iterrows():
        prompt = f"{row["short description"]}, associated with: {", ".join(row["tags"])}"
        result.append(prompt)
    return result

prompts = form_prompts()
embeddings = model.encode(prompts, convert_to_tensor=True)
type(embeddings)
embeddings.shape

torch.Size([5034, 384])

In [25]:
query = "december"
start_time = time.time()
query_embedding = model.encode(query, convert_to_tensor=True)

from sentence_transformers import util
cosine_scores = util.cos_sim(query_embedding, embeddings)
cosine_scores_list = cosine_scores[0].tolist()
top_results = sorted(range(len(cosine_scores_list)), key=lambda i: cosine_scores_list[i], reverse=True)[:5]

print("Top 5 most similar emojis to '{}'".format(query))

for idx in top_results:
    print("{}: {} (Score: {:.4f})".format(df.iloc[idx]["character"], df.iloc[idx]["short description"], cosine_scores_list[idx]))
end_time = time.time()
print("Search took {:.4f} seconds".format(end_time - start_time))

Top 5 most similar emojis to 'december'
⛄: SNOWMAN WITHOUT SNOW (Score: 0.4025)
🎅🏿: SANTA CLAUS DARK SKIN TONE (Score: 0.3294)
🧑🏿‍🎄: MX CLAUS DARK SKIN TONE (Score: 0.3232)
☃️: SNOWMAN (Score: 0.3139)
☃: SNOWMAN (Score: 0.3139)
Search took 0.0472 seconds


In [42]:
index = pd.read_json("emoji.json")

skin_tone_modifiers = [
    '1F3FB',
    '1F3FC',
    '1F3FD',
    '1F3FE',
    '1F3FF',
]
skipped = {}
skin_tone = 0

images = []
for _, row in df.iterrows():
    unified = row['unicode'][2:].replace(' ', '-')
    if any(modifier in unified for modifier in skin_tone_modifiers):
        skin_tone += 1
        # print it with 10% chance
        if time.time() % 10 < 1:
            print("Skipping emoji with skin tone modifier:", row['character'], "with unified code:", unified)   
        continue

    # find row in index with matching unified code
    matched_row = index[index['unified'] == unified]
    if not matched_row.empty:
        matched_row = matched_row.iloc[0]
        # using twemoji, which has no missing images

        sheet_x, sheet_y = matched_row['sheet_x'], matched_row['sheet_y']
        sheet_size = 64 + 2 # 64x64 images with 2px padding
        left = sheet_x * sheet_size + 1
        upper = sheet_y * sheet_size + 1
        right = left + 64
        lower = upper + 64
        # emoji_image = sheet.crop((left, upper, right, lower)).convert("RGB")
        # images.append(emoji_image)
    else:
        matched_row = index[index['non_qualified'] == unified]
        if not matched_row.empty:
            pass
            # print("Found non-qualified match for emoji:", row['character'], "with unified code:", unified)
        else:
            pass
            # skiped[unified] = skiped.get(unified, 0) + 1
            # print("No image found for emoji:", row['character'], "with unified code:", unified)

# print("Total emojis with no image found:", not_found)
print("Total emojis skipped due to skin tone modifiers:", skin_tone)

Skipping emoji with skin tone modifier: 🤶🏿 with unified code: 1F936-1F3FF
Skipping emoji with skin tone modifier: 🤶🏻 with unified code: 1F936-1F3FB
Skipping emoji with skin tone modifier: 🤶🏾 with unified code: 1F936-1F3FE
Skipping emoji with skin tone modifier: 🤶🏼 with unified code: 1F936-1F3FC
Skipping emoji with skin tone modifier: 🤶🏽 with unified code: 1F936-1F3FD
Skipping emoji with skin tone modifier: 👌🏿 with unified code: 1F44C-1F3FF
Skipping emoji with skin tone modifier: 👌🏻 with unified code: 1F44C-1F3FB
Skipping emoji with skin tone modifier: 👌🏾 with unified code: 1F44C-1F3FE
Skipping emoji with skin tone modifier: 👌🏼 with unified code: 1F44C-1F3FC
Skipping emoji with skin tone modifier: 👌🏽 with unified code: 1F44C-1F3FD
Skipping emoji with skin tone modifier: 🎅🏿 with unified code: 1F385-1F3FF
Skipping emoji with skin tone modifier: 🎅🏻 with unified code: 1F385-1F3FB
Skipping emoji with skin tone modifier: 🎅🏾 with unified code: 1F385-1F3FE
Skipping emoji with skin tone modifier